## Growing season classification (4GST) -- ctl and aff

Computes, for every model, the 4GST growing-season classification (onset month, offset month, phenology type) from each scenario's own LAI climatology, for both the control (ssp370) and afforestation (ssp126Lu) runs. Produces two kinds of output, both consumed by other notebooks in `code/revision`:

1. The raw per-pixel classification (onset, offset, gst_type), cached to `./cache/growing_season/` for any notebook that needs to build its own time-resolved growing-season mask (see `lumip_utils.growing_season_mask` / `load_growing_season`) -- e.g. an interannual regression that must keep the time dimension.
2. Ready-made 30-year (2071-2100) growing-season-**mean** maps for aff, ctl and their difference (`xr_aff_gs`, `xr_ctl_gs`, `xr_delta_gs`), saved to `data_path` as `xr_{aff,ctl,delta}_gs_<model>.nc` -- this is what `seb_grow_seas.ipynb` loads directly (see its "Import Growing season data" section).

**Run this notebook first**, and run it (and every other notebook in this folder) with its working directory set to `code/revision/` -- the `./cache/...` path is relative, and Jupyter's default working directory is wherever the notebook file lives, so opening these notebooks directly from `code/revision/` just works. (Retitled from the original copy-pasted "SEB and IBPM analysis" header, which was a leftover from whatever notebook this one was originally cloned from and never matched this notebook's actual content.)

In [1]:
import os
import glob

import math
import matplotlib as mpl
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy as cart
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
from cartopy.util import add_cyclic_point
import seaborn as sns

import numpy as np
import xarray as xr
import cftime
import pandas as pd

import dask

In [2]:
# --- shared, corrected helper functions (see code/revision/lumip_utils.py) ---
# compute_4gst: fixes the evergreen-threshold bug in the old classify_phenology
#   (amp/lai_mean, not amp/lai_min).
# growing_season_mask: replaces the old growing_season()/phen_gs(), which
#   collapsed the growing season to a single nanmean scalar and discarded the
#   time dimension. This returns a (time, lat, lon) boolean mask instead, so
#   `.where(...)` only NaNs out the non-growing-season months and the full
#   time-resolved series survives for downstream trend/regression analysis.
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from lumip_utils import compute_4gst, growing_season_mask, gs_cache_path


In [3]:
# NOTE (revision): data_path/out_path for "locale_mc" fixed from the stale
# .../Desktop/afforestation_LUMIP/... in the original to .../Documents/afforestation_LUMIP/...
# -- that's where the data actually lives; the original path would have failed on the
# very first data-loading cell below.
work = "locale_mc" # or "levante"

if work == "levante":
    data_path = "/home/m/m301093/data/clean/"
    out_path = "/home/m/m301093/results_biophys/"
if work == "locale_wnd":
    data_path = "G:/My Drive/MPIM/data/"
    out_path = "G:/My Drive/MPIM/Figures/"
if work == "locale_mc":
    data_path = "/Users/matteomastropierro/Documents/afforestation_LUMIP/data/"
    out_path = "/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/" \
    "Figures/"

## Functions

In [4]:
def xr_multimodel_sign(xr_in, models):
    # Computes the Multimodel MEAN and STD
    # xr_in: list of xarray models
    mm = xr.concat(xr_in, dim = "esm", coords = "minimal", compat = "override")
    
    std = xr.concat(xr_in, dim = "esm", coords = "minimal", compat = "override").std(dim = "esm")
    
    # Multimodel agreement in the sign of the sensitivity
    sign = xr.where(mm > 0, 1, -1)
    agreement = (np.abs(sign.sum(dim = "esm")/len(models)))
    
    mean = mm.mean(dim = "esm")
    return (mean, std, agreement)

In [5]:
def get_pixel_area(ds):
    """Calculate the area of each pixel in m^2 given latitudes and longitudes."""
    lat = ds.lat.values
    lon = ds.lon.values
    # Earth radius in meters
    R = 6.371e6
    rad = (2 * math.pi / 360)  # Convert degrees to radians

    # Reshape lat and lon to make sure they have the correct shape
    if lat[0] > lat[-1]:
        lat = lat[::-1]

    # Calculate the area of each pixel
    pixel_areas = np.zeros((len(lat) - 1, len(lon) - 1))
    for i in range(len(lat) - 1):
        for j in range(len(lon) - 1):
            pixel_areas[i, j] = (R ** 2 *
                                 (math.sin(rad * lat[i + 1]) - math.sin(rad * lat[i])) *
                                 abs(lon[j + 1] - lon[j]) * rad)

    pixel_areas = xr.DataArray(pixel_areas, coords=[lat[:-1], lon[:-1]], dims=["lat", "lon"])
    return pixel_areas

def cell_weight(ds):
    R = 6.371e6
    dϕ = np.deg2rad(ds.lat[1] - ds.lat[0])
    dλ = np.deg2rad(ds.lon[1] - ds.lon[0])
    dlat = R * dϕ * xr.ones_like(ds.lon)
    dlon = R * dλ * np.cos(np.deg2rad(ds.lat))
    cell_area = dlon * dlat
    return(cell_area)

def lon180(ds):
    ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
    ds = ds.sortby(ds.lon)
    return ds

In [6]:
from scipy.stats import mannwhitneyu

def mann_whitney_pvalue(arr1, arr2):
    stat, p = mannwhitneyu(arr1, arr2, alternative="two-sided")
    return p

def xr_multipletest(p, alpha=0.05, method='fdr_bh', **multipletests_kwargs):
    """Apply statsmodels.stats.multitest.multipletests for multi-dimensional xr.objects."""
    from statsmodels.stats.multitest import multipletests
    # stack all to 1d array
    p_stacked = p.stack(s=p.dims)
    # mask only where not nan: https://github.com/statsmodels/statsmodels/issues/2899
    mask = np.isfinite(p_stacked)
    pvals_corrected = np.full(p_stacked.shape, np.nan)
    reject = np.full(p_stacked.shape, np.nan)
    # apply test where mask
    reject[mask] = multipletests(
        p_stacked[mask], alpha=alpha, method=method, **multipletests_kwargs)[0]
    pvals_corrected[mask] = multipletests(
        p_stacked[mask], alpha=alpha, method=method, **multipletests_kwargs)[1]

    def unstack(reject, p_stacked):
        """Exchange values from p_stacked with reject (1d array) and unstack."""
        xreject = p_stacked.copy()
        xreject.values = reject
        xreject = xreject.unstack()
        return xreject

    reject = unstack(reject, p_stacked)
    pvals_corrected = unstack(pvals_corrected, p_stacked)
    return reject, pvals_corrected


In [7]:
def lag_linregress_3D(x, y, lagx=0, lagy=0):
    """
    Input: Two xr.Datarrays of any dimensions with the first dim being time. 
    Thus the input data could be a 1D time series, or for example, have three dimensions (time,lat,lon). 
    Datasets can be provied in any order, but note that the regression slope and intercept will be calculated
    for y with respect to x.
    Output: Covariance, correlation, regression slope and intercept, p-value, and standard error on regression
    between the two datasets along their aligned time dimension.  
    Lag values can be assigned to either of the data, with lagx shifting x, and lagy shifting y, with the specified lag amount. 
    """ 
    #1. Ensure that the data are properly alinged to each other. 
    x,y = xr.align(x,y)
    
    #2. Add lag information if any, and shift the data accordingly
    if lagx!=0:
        #If x lags y by 1, x must be shifted 1 step backwards. 
        #But as the 'zero-th' value is nonexistant, xr assigns it as invalid (nan). Hence it needs to be dropped
        x   = x.shift(time = -lagx).dropna(dim='time')
        #Next important step is to re-align the two datasets so that y adjusts to the changed coordinates of x
        x,y = xr.align(x,y)

    if lagy!=0:
        y   = y.shift(time = -lagy).dropna(dim='time')
        x,y = xr.align(x,y)
 
    #3. Compute data length, mean and standard deviation along time axis for further use: 
    n     = x.shape[0]
    xmean = x.mean(axis=0)
    ymean = y.mean(axis=0)
    xstd  = x.std(axis=0)
    ystd  = y.std(axis=0)
    
    #4. Compute covariance along time axis
    cov   =  np.sum((x - xmean)*(y - ymean), axis=0)/(n)
    
    #5. Compute correlation along time axis
    cor   = cov/(xstd*ystd)
    
    #6. Compute regression slope and intercept:
    slope     = cov/(xstd**2)
    intercept = ymean - xmean*slope
    y_pred =  intercept + slope*x
    res = y - y_pred

    #7. Compute P-value and standard error
    #Compute t-statistics
    tstats = cor*np.sqrt(n-2)/np.sqrt(1-cor**2)
    stderr = slope/tstats
    
    from scipy.stats import t
    pval   = t.sf(tstats, n-2)*2
    pval   = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    #return cov,cor,slope,intercept,pval,stderr
    results = {}
    results["cor"] = cor
    results["coef"] = slope
    results["pval"] = pval
    return results

## Import preprocessed data

In [8]:
models = [ "ACCESS-ESM1-5", "CMCC-ESM2", "CESM2", "CanESM5", "IPSL-CM6A-LR", "MPI-ESM1-2-LR", "UKESM1-0-LL"]

In [9]:
scenario = 'ssp126Lu'
xr_aff = []
for i,mm in enumerate(models):
    filepath = glob.glob(os.path.join(data_path + 'xr_' + scenario + "_" + mm + '.nc'))[0]                                       ## List of files sorted by name
    content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
    content = content.assign_attrs(esm = mm)
    xr_aff.append(content)


scenario = 'ssp370'
xr_ctl = []
for i,mm in enumerate(models):
    filepath = glob.glob(os.path.join(data_path + 'xr_' + scenario + "_" + mm + '.nc'))[0]                                       ## List of files sorted by name
    content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
    content = content.assign_attrs(esm = mm)
    xr_ctl.append(content)


/var/folders/h1/tgvjn96s32q5lfh32xvkh1hh0000gp/T/ipykernel_11374/444337245.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 240. This could degrade performance. Instead, consider rechunking after loading.
  content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
/var/folders/h1/tgvjn96s32q5lfh32xvkh1hh0000gp/T/ipykernel_11374/444337245.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 240. This could degrade performance. Instead, consider rechunking after loading.
  content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
/var/folders/h1/tgvjn96s32q5lfh32xvkh1hh0000gp/T/ipykernel_11374/444337245.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 240. This could degrade perfo

In [10]:
scenario = 'ssp126Lu'
xr_aff_pft = []
for i,mm in enumerate(models):
    filepath = glob.glob(os.path.join(data_path + 'xr_' + scenario + "_" + mm + '_pft.nc'))[0]                                       ## List of files sorted by name
    content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
    content = content.assign_attrs(esm = mm)
    xr_aff_pft.append(content)


scenario = 'ssp370'
xr_ctl_pft = []
for i,mm in enumerate(models):
    filepath = glob.glob(os.path.join(data_path + 'xr_' + scenario + "_" + mm + '_pft.nc'))[0]                                       ## List of files sorted by name
    content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
    content = content.assign_attrs(esm = mm)
    xr_ctl_pft.append(content)


/var/folders/h1/tgvjn96s32q5lfh32xvkh1hh0000gp/T/ipykernel_11374/3490561412.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 240. This could degrade performance. Instead, consider rechunking after loading.
  content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
/var/folders/h1/tgvjn96s32q5lfh32xvkh1hh0000gp/T/ipykernel_11374/3490561412.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 240. This could degrade performance. Instead, consider rechunking after loading.
  content = xr.open_dataset(filepath, drop_variables=["time_bnds","lon_bnds","lat_bnds"],engine = 'netcdf4',chunks={"time": 240})
/var/folders/h1/tgvjn96s32q5lfh32xvkh1hh0000gp/T/ipykernel_11374/3490561412.py:5: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 240. This could degrade pe

### Data homogenization

In [11]:
# CanESM5 does not provide Latent Heat
# We derive it as Rn - H (ignoring Ground Heat fluxes that are negligible over land)

xr_ctl[3]["hfls"] = (xr_ctl[3].rsds - xr_ctl[3].rlds) - xr_ctl[3].hfss
xr_aff[3]["hfls"] = (xr_aff[3].rsds - xr_aff[3].rlds) - xr_aff[3].hfss

# Rename ta to tas 
xr_aff[3] = xr_aff[3].rename({"ta":"tas"})
xr_ctl[3] = xr_ctl[3].rename({"ta":"tas"})

# Select the first pressure level
xr_aff[3] = xr_aff[3].isel(plev = 0)
xr_ctl[3] = xr_ctl[3].isel(plev = 0)

# Try to interpolate surface temperature for CanESM5
xr_ctl[3]["tas"] = (xr_ctl[3].ts + xr_ctl[3].tas)/2
xr_aff[3]["tas"] = (xr_aff[3].ts + xr_aff[3].tas)/2


In [12]:
# Convert pr and evspsbl from kg m-2 s-1 to mm month-1
for i,mm in enumerate(models):
    xr_ctl[i]["pr"] = xr_ctl[i].pr*86400* xr_ctl[i].time.dt.days_in_month
    xr_aff[i]["pr"] = xr_aff[i].pr*86400* xr_aff[i].time.dt.days_in_month
    
    xr_ctl[i]["pr"].attrs=dict(units="mm/months")
    xr_aff[i]["pr"].attrs=dict(units="mm/months")

    if mm == "CMCC-ESM2" or "CESM2": # LE in W*m^2; λ is in MJ kg-1 = MW*s kg-1 
        # https://earthscience.stackexchange.com/questions/20733/fluxnet15-how-to-convert-latent-heat-flux-to-actual-evapotranspiration
        xr_ctl[i]["evspsbl"] = xr_ctl[i].hfls/(2.501 - (2.361e-3) * (xr_ctl[i].tas-273.15))*10**6
        xr_aff[i]["evspsbl"] = xr_aff[i].hfls/(2.501 - (2.361e-3) * (xr_aff[i].tas-273.15))*10**6

    else:
        None
    
    xr_ctl[i]["evspsbl"] = xr_ctl[i].evspsbl*86400* xr_ctl[i].time.dt.days_in_month
    xr_aff[i]["evspsbl"] = xr_aff[i].evspsbl*86400* xr_aff[i].time.dt.days_in_month
    
    xr_ctl[i]["evspsbl"].attrs=dict(units="mm/months")
    xr_aff[i]["evspsbl"].attrs=dict(units="mm/months")
    
    xr_aff[i]["albedo"] = xr_aff[i].rsus / xr_aff[i].rsds
    xr_ctl[i]["albedo"] = xr_ctl[i].rsus / xr_ctl[i].rsds
    
    

In [13]:
# Homogenize lat and lon
for i,mm in enumerate(models):
    content = xr_aff[i]
    xr_aff_pft[i]["lat"] = content.tas.lat; xr_aff_pft[i]["lon"] = content.lon

    content = xr_ctl[i]
    xr_ctl_pft[i]["lat"] = content.tas.lat; xr_ctl_pft[i]["lon"] = content.lon

## Growing season

In [14]:
xr_delta = []
xr_delta_pft = []
for i,mm in enumerate(models):
    xr_delta.append(xr_aff[i] - xr_ctl[i])
    xr_delta_pft.append(xr_aff_pft[i] - xr_ctl_pft[i])

In [15]:
# classify_phenology() removed: superseded by lumip_utils.compute_4gst(), which
# fixes the evergreen-classification bug (see lumip_utils.py docstring / FIX 2).


In [16]:
# Growing-season-type (4GST) classification, computed independently for both
# the control (ssp370, xr_ctl) and afforestation (ssp126Lu, xr_aff) runs, from
# each model's whole-record monthly LAI climatology (same scope as the
# original -- no period restriction was applied here before, and none is
# added now). Cached to disk so every other notebook in code/revision can load
# it via lumip_utils.load_growing_season(models, scenario=...) instead of
# recomputing its own (potentially inconsistent) phenology classification.
xr_phen = {"ctl": [], "aff": []}
_scenario_ds = {"ctl": xr_ctl, "aff": xr_aff}

for scenario, ds_list in _scenario_ds.items():
    for m, mm in enumerate(models):
        lai_monthly = ds_list[m].lai.groupby("time.month").mean("time").load()
        phen = compute_4gst(lai_monthly, month_dim="month")
        phen = phen.assign_coords(lon=lai_monthly.lon, lat=lai_monthly.lat)
        xr_phen[scenario].append(phen)

os.makedirs("./cache/growing_season", exist_ok=True)
for scenario in ("ctl", "aff"):
    for m, mm in enumerate(models):
        xr_phen[scenario][m].to_netcdf(gs_cache_path(scenario, mm))
print("Growing-season phenology cached to ./cache/growing_season/ for ctl and aff, "
      f"{len(models)} models each.")


Growing-season phenology cached to ./cache/growing_season/ for ctl and aff, 7 models each.


**Which run defines the growing-season window used below and by every other revision notebook.** `"ctl"` (recommended, default) keeps the same seasonal window in both runs, so the aff-ctl difference is not itself contaminated by a shifted growing season; `"aff"` is available as a robustness check only. This must be set identically wherever `growing_season_mask`/`load_growing_season` is used downstream -- it is not itself cached, only the ctl and aff phenology are.

In [17]:
GS_REFERENCE = "ctl"


In [18]:
from lumip_utils import growing_season_reduce

phen_delta = xr_phen[GS_REFERENCE]
xr_delta_gs = []
for m, mm in enumerate(models):
    delta = xr_delta[m].sel(time=slice("2071", "2100"))
    months = delta.time.dt.month.values
    onset = phen_delta[m].gst_start_month
    offset = phen_delta[m].gst_end_month

    delta_results = {}
    for var in delta.data_vars:
        if delta[var].dtype.kind == "M":  # skip datetime64 vars (e.g. IPSL time_bounds)
            continue
        series = delta[var].load()
        delta_results[var] = xr.apply_ufunc(
            growing_season_reduce, series, months, onset, offset,
            input_core_dims=[["time"], ["time"], [], []],
            output_core_dims=[[]], vectorize=True, output_dtypes=[float],
        )
    xr_delta_gs.append(xr.Dataset(delta_results))


/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning: Mean of empty slice
  return np.nanmean(series)
/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning: Mean of empty slice
  return np.nanmean(series)
/Users/matteomastropierro/miniconda/envs/geo_clim_py3.12/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning: Mean of empty slice
  return np.nanmean(series)
/Users/matteomastropierro/miniconda/envs/geo_clim_py3.12/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning

In [19]:
phen_ref = xr_phen[GS_REFERENCE]
xr_aff_gs = []
xr_ctl_gs = []

for m, mm in enumerate(models):
    aff = xr_aff[m].sel(time=slice("2071", "2100"))
    ctl = xr_ctl[m].sel(time=slice("2071", "2100"))
    months = aff.time.dt.month.values.astype(int)
    onset = phen_ref[m].gst_start_month
    offset = phen_ref[m].gst_end_month

    aff_results = {}
    ctl_results = {}
    for var in aff.data_vars:
        if aff[var].dtype.kind == "M":  # skip datetime64 vars (e.g. IPSL time_bounds)
            continue
        series = aff[var].load()
        aff_results[var] = xr.apply_ufunc(
            growing_season_reduce, series, months, onset, offset,
            input_core_dims=[["time"], ["time"], [], []],
            output_core_dims=[[]], vectorize=True, output_dtypes=[float],
        )
    for var in ctl.data_vars:
        if ctl[var].dtype.kind == "M":
            continue
        series = ctl[var].load()
        ctl_results[var] = xr.apply_ufunc(
            growing_season_reduce, series, months, onset, offset,
            input_core_dims=[["time"], ["time"], [], []],
            output_core_dims=[[]], vectorize=True, output_dtypes=[float],
        )

    xr_aff_gs.append(xr.Dataset(aff_results))
    xr_ctl_gs.append(xr.Dataset(ctl_results))


/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning: Mean of empty slice
  return np.nanmean(series)
/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning: Mean of empty slice
  return np.nanmean(series)
/Users/matteomastropierro/miniconda/envs/geo_clim_py3.12/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning: Mean of empty slice
  return np.nanmean(series)
/Users/matteomastropierro/Documents/afforestation_LUMIP/Submission/code/revision/lumip_utils.py:352: RuntimeWarning: Mean of empty slice
  return np.nanmean(series)
/Users/matteomastropierro/miniconda/envs/geo_clim_py3.12/lib/python3.12/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encou

In [20]:
# All three are consumed by seb_grow_seas.ipynb's "Import Growing season data"
# section (xr_{aff,ctl,delta}_gs_<model>.nc, loaded from data_path). The
# original here (a) only saved xr_ctl_gs (aff/delta were commented out) and
# (b) saved to out_path, not data_path -- so seb_grow_seas.ipynb's loader could
# never actually find these files. Both are fixed here.
for i, mm in enumerate(models):
    xr_aff_gs[i].to_netcdf(data_path + "/xr_aff_gs_" + mm + ".nc")
    xr_ctl_gs[i].to_netcdf(data_path + "/xr_ctl_gs_" + mm + ".nc")
    xr_delta_gs[i].to_netcdf(data_path + "/xr_delta_gs_" + mm + ".nc")
